In [6]:
# 필요한 라이브러리 설치
%pip install torch torchvision matplotlib

Note: you may need to restart the kernel to use updated packages.


# 라이브러리 준비 및 설정

In [7]:
import platform

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용 장치:", device)

사용 장치: cpu


# 전역 변수 지정

In [8]:
# 분류(10종류의 이미지)
CLASSES = ["티셔츠", "바지", "스웨터", "드레스", "코트", "샌들", "셔츠", "운동화", "가방", "앵클부츠"]
MODEL_FILE = "cnn_fashion.pt"

# 1. 데이터 선택 및 사전 준비

In [9]:
# ---------- 1. 데이터 선택 + 2. 사전 준비 ----------
# ToTensor: 사진(0~255) → 텐서(0~1), 모양 (채널 1, 세로 28, 가로 28)
# 채널 : 1 => 흑백
# 채널 : 3 => 칼라
# Normalize: 0~1 → -1~1 로 가운데를 0에 맞춘다 (스케일링과 같은 이유로 학습이 안정된다)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_full = datasets.FashionMNIST("data", train=True, download=False, transform=transform)
test_data = datasets.FashionMNIST("data", train=False, download=False, transform=transform)
print("학습용:", len(train_full), "시험용:", len(test_data))

image, label = train_full[0] # 첫번째 이미지 가져옴
print("사진 한 장의 모양:", image.shape, "정답:", CLASSES[label])


학습용: 60000 시험용: 10000
사진 한 장의 모양: torch.Size([1, 28, 28]) 정답: 앵클부츠


In [10]:
# 데이터 먼저 보기
# fig, axes = plt.subplots(2, 4, figsize=(12, 3.5))
# for i, ax in enumerate(axes.flat):
#     img, lbl = train_full[i]
#     ax.imshow(img[0], cmap="gray")
#     ax.set_title(CLASSES[lbl], fontsize=9)
#     ax.axis("off")
# plt.tight_layout()
# plt.show()

# 훈련/테스트 데이터 분리

In [11]:
# ---------- 3. 훈련, 검증, 테스트 분리 ----------
# 시험용 1만 장은 이미 따로 있다. 학습용 6만 장에서 5천 장을 검증용으로 떼어 둔다.
train_data, valid_data = random_split(train_full, [55000, 5000], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)   # 128장씩, 에포크마다 섞어서
valid_loader = DataLoader(valid_data, batch_size=256)
test_loader = DataLoader(test_data, batch_size=256)

# 학습

In [ ]:
# ---------- 4. 학습 ----------
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 특징 찾기: 합성곱(패턴 찾기) → ReLU(꺾기) → 풀링(크기 줄이기) 을 두 번
        self.features = nn.Sequential(
            # kernel_size 패턴 찾는 돋보기를 3px x 3px로 선택
            # 32 : 패턴의 종류
            # padding : 1, 이미지 위, 아래, 좌, 우에 여백을 추가
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 2x2에서 가장 큰 값만 → (32, 14, 14)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # 앞의 패턴을 조합해 더 큰 패턴 64종류 → (64, 14, 14)
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → (64, 7, 7)
        )
        # 판단하기: 찾은 특징을 한 줄로 펴서 MLP로 10종류 중 하나를 고른다
        self.classifier = nn.Sequential(
            nn.Flatten(),                                  # (64, 7, 7) → 3136
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),                               # 외우기 방지
            nn.Linear(128, 10),                            # 출력층 10 (옷 종류)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = FashionCNN().to(device)
print(model)
print("학습할 숫자:", sum(p.numel() for p in model.parameters()))